### JSON Parsing ###

In [23]:
import os
import json

os.makedirs("data/json_files", exist_ok=True)

json_data = {
    "company": "Example Corp",
    "employees": [
        {
            "name": "Alice",
            "age": 30,
            "department": "HR",
            "skills": [
                "Recruitment",
                "Employee Relations"
            ],
            "projects": [
                {
                    "name": "Onboarding Program",
                    "status": "Completed"
                },
                {
                    "name": "Employee Engagement Survey",
                    "status": "In Progress"
                }
            ]
        },
        {
            "name": "Bob",
            "age": 25,
            "department": "Engineering",
            "skills": [
                "Software Development",
                "Testing"
            ],
            "projects": [
                {
                    "name": "API Development",
                    "status": "In Progress"
                }
            ]
        },
        {
            "name": "Charlie",
            "age": 35,
            "department": "Marketing",
            "skills": [
                "Content Creation",
                "Social Media Management"
            ],
            "projects": [
                {
                    "name": "Brand Campaign",
                    "status": "Completed"
                }
            ]
        }
    ]
}

In [24]:
with open("data/json_files/company_data.json", "w") as json_file:
    json.dump(json_data,json_file, indent=4)

In [25]:
json1_data = [{
    "time": "2023-01-01T12:00:00Z",
    "event": "UserLogin",
    "user_id": "user123"
},
{
    "time": "2023-01-01T12:05:00Z",
    "event": "PageView",
    "user_id": "user123",
    "page": "/home"
},
{
    "time": "2023-01-01T12:10:00Z",
    "event": "ButtonClick",
    "user_id": "user123",
    "button_id": "signup_button"
}]

with open("data/json_files/user_activity.json", "w") as json_file:
    for item in json1_data:
        json_file.write(json.dumps(item) + "\n")

### JSON Processing ###

In [26]:
from langchain_community.document_loaders import JSONLoader
import json
#method 1 : jsonloader with jq_schema
employee_loader = JSONLoader(file_path="data/json_files/company_data.json", jq_schema=".employees[]",text_content=False)
employee_docs = employee_loader.load()
print(f"Number of employee documents loaded: {len(employee_docs)}")
print(f"First employee document: {employee_docs[0].page_content}")
print(f"Metadata of first employee document: {employee_docs[0].metadata}")
print(employee_docs)


Number of employee documents loaded: 3
First employee document: {"name": "Alice", "age": 30, "department": "HR", "skills": ["Recruitment", "Employee Relations"], "projects": [{"name": "Onboarding Program", "status": "Completed"}, {"name": "Employee Engagement Survey", "status": "In Progress"}]}
Metadata of first employee document: {'source': 'D:\\RAG\\RAG-Learning\\Repo-1\\0-DataIngestParsing\\data\\json_files\\company_data.json', 'seq_num': 1}
[Document(metadata={'source': 'D:\\RAG\\RAG-Learning\\Repo-1\\0-DataIngestParsing\\data\\json_files\\company_data.json', 'seq_num': 1}, page_content='{"name": "Alice", "age": 30, "department": "HR", "skills": ["Recruitment", "Employee Relations"], "projects": [{"name": "Onboarding Program", "status": "Completed"}, {"name": "Employee Engagement Survey", "status": "In Progress"}]}'), Document(metadata={'source': 'D:\\RAG\\RAG-Learning\\Repo-1\\0-DataIngestParsing\\data\\json_files\\company_data.json', 'seq_num': 2}, page_content='{"name": "Bob", "

In [27]:
from typing import List
from langchain_core.documents import Document
print("\n\nMethod 2: Using JSONLoader with custom parsing function")

def process_json_intelligently(filepath:str)->List[Document]:
   with open(filepath, 'r') as file:
       data = json.load(file)

   documents = []
   for employee in data.get("employees", []):
       content = f"""Employee Profile: 
       Name: {employee.get('name')}
       Age: {employee.get('age')}
       Department: {employee.get('department')}
       Skills: {', '.join(employee.get('skills', []))}
 
       Projects: """
       for project in employee.get('projects', []):
           content += f"\n  - {project.get('name')} (Status: {project.get('status')})"

       doc = Document(page_content=content, metadata={"source": filepath, "employee_name": employee.get('name'), "department": employee.get('department')})
       documents.append(doc)
   return documents




Method 2: Using JSONLoader with custom parsing function


In [28]:
processed_docs = process_json_intelligently("data/json_files/company_data.json")
print(processed_docs)
print(processed_docs[0].page_content)

[Document(metadata={'source': 'data/json_files/company_data.json', 'employee_name': 'Alice', 'department': 'HR'}, page_content='Employee Profile: \n       Name: Alice\n       Age: 30\n       Department: HR\n       Skills: Recruitment, Employee Relations\n\n       Projects: \n  - Onboarding Program (Status: Completed)\n  - Employee Engagement Survey (Status: In Progress)'), Document(metadata={'source': 'data/json_files/company_data.json', 'employee_name': 'Bob', 'department': 'Engineering'}, page_content='Employee Profile: \n       Name: Bob\n       Age: 25\n       Department: Engineering\n       Skills: Software Development, Testing\n\n       Projects: \n  - API Development (Status: In Progress)'), Document(metadata={'source': 'data/json_files/company_data.json', 'employee_name': 'Charlie', 'department': 'Marketing'}, page_content='Employee Profile: \n       Name: Charlie\n       Age: 35\n       Department: Marketing\n       Skills: Content Creation, Social Media Management\n\n       P

In [38]:
### Loading user_activity.json into metadata ###

def process_user_activity_json(filepath:str)->List[Document]:
    with open(filepath, 'r') as file:
         documents = []
         for line in file:
              data = json.loads(line)              
              doc = Document(page_content="", metadata={"Time": data.get('time'), "event": data.get('event'), "user_id": data.get('user_id')})
              documents.append(doc)
    return documents

In [39]:
activity_docs = process_user_activity_json("data/json_files/user_activity.json")
print(f"Number of processed documents: {len(activity_docs)}")
print(f"Metadata of first processed document: {activity_docs[0].metadata}")

Number of processed documents: 3
Metadata of first processed document: {'Time': '2023-01-01T12:00:00Z', 'event': 'UserLogin', 'user_id': 'user123'}
